In [1]:
!pip install pandas numpy nltk gensim scikit-learn

In [2]:
#Load and Combine Dataset
import pandas as pd

fake = pd.read_csv("Fake.csv")
true = pd.read_csv("True.csv")

fake['label'] = 0   # Fake
true['label'] = 1   # Real

df = pd.concat([fake, true])
df = df[['text', 'label']]

df = df.sample(frac=1).reset_index(drop=True)  # shuffle
df.head()

,text,label
0,"21st Century Wire says Since the late 1960 s, ...",0
1,ROME (Reuters) - Lebanese Prime Minister Saad ...,1
2,"21st Century Wire says So far, after nearly 20...",0
3,"On Sunday, thousands of protesters came togeth...",0
4,Barbra Streisand was an Obama sycophant and on...,0


In [3]:
#Text Preprocessing Lowercase, Remove punctuation, Remove stopwords, Lemmatization
import nltk
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text)
    
    words = text.split()
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]
    
    return words

df['tokens'] = df['text'].apply(preprocess)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\GOWTHAMSANJAY\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\GOWTHAMSANJAY\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [5]:
#Train Word2Vec Model
from gensim.models import Word2Vec

w2v_model = Word2Vec(
    sentences=df['tokens'],
    vector_size=100,
    window=5,
    min_count=2,
    workers=4
)

In [6]:
#Convert Text to Vectors
import numpy as np

def get_avg_vector(tokens, model, size):
    vec = np.zeros(size)
    count = 0
    
    for word in tokens:
        if word in model.wv:
            vec += model.wv[word]
            count += 1
    
    if count != 0:
        vec = vec / count
    
    return vec

X = np.array([get_avg_vector(tokens, w2v_model, 100) for tokens in df['tokens']])
y = df['label']

In [7]:
#Train-Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [8]:
#Train Model
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train, y_train)

LogisticRegression()

In [9]:
#Evaluate Model
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.97728285077951
              precision    recall  f1-score   support

           0       0.98      0.98      0.98      4704
           1       0.97      0.98      0.98      4276

    accuracy                           0.98      8980
   macro avg       0.98      0.98      0.98      8980
weighted avg       0.98      0.98      0.98      8980



In [10]:
#Test Your Model
def predict_news(text):
    tokens = preprocess(text)
    vec = get_avg_vector(tokens, w2v_model, 100).reshape(1, -1)
    pred = model.predict(vec)
    
    return "Fake News" if pred[0] == 0 else "Real News"

print(predict_news("Breaking news: New government policy announced"))

Fake News


In [11]:
print(predict_news("The BBC (British Broadcasting Corporation) is the United Kingdom’s national public service broadcaster."))

Real News
